# Bank Transaction Fraud Detection System
### Advanced Machine Learning Pipeline · End-to-End

---

> **Dataset:** `Bank_Transaction_Fraud_Detection.csv`  
> **Records:** 200,000 transactions · **Features:** 24 raw → 32 engineered  
> **Target:** `Is_Fraud` (binary) · **Fraud Rate:** ~5.04%  
> **Models:** Logistic Regression · Random Forest · XGBoost · Threshold Tuning  
> **Techniques:** SMOTE · Cross-Validation · SHAP · Risk Scoring · Feature Engineering

---

In [1]:
# Install any missing packages 
import subprocess, sys

REQUIRED = ["scikit-learn", "xgboost", "imbalanced-learn", "shap",
            "matplotlib", "seaborn", "pandas", "numpy"]

for pkg in REQUIRED:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("  All packages available.")

c:\Users\USER\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  All packages available.


## Import Libraries

In [2]:
# ── Core imports ──────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib.pyplot   as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches  as mpatches
import seaborn             as sns

# Sklearn — preprocessing
from sklearn.model_selection   import (train_test_split, StratifiedKFold,
                                        cross_val_score, learning_curve)
from sklearn.preprocessing     import LabelEncoder, StandardScaler, label_binarize
from sklearn.pipeline          import Pipeline

# Sklearn — models
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier

# Sklearn — metrics
from sklearn.metrics           import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score,
    ConfusionMatrixDisplay
)

# XGBoost & SMOTE
from xgboost              import XGBClassifier
from imblearn.over_sampling import SMOTE

# SHAP for interpretability
import shap

# ── Global plot style ─────────────────────────────────────────────────────────
DARK = {
    "bg"     : "#0F1117",
    "panel"  : "#1A1D27",
    "grid"   : "#252836",
    "text"   : "#E8EAF0",
    "sub"    : "#8890A4",
    "blue"   : "#4F8EF7",
    "red"    : "#F7614F",
    "green"  : "#4FD1A5",
    "gold"   : "#F7C84F",
    "purple" : "#A78BFA",
}

plt.rcParams.update({
    "figure.facecolor"  : DARK["bg"],
    "axes.facecolor"    : DARK["panel"],
    "axes.edgecolor"    : DARK["grid"],
    "axes.labelcolor"   : DARK["text"],
    "xtick.color"       : DARK["sub"],
    "ytick.color"       : DARK["sub"],
    "text.color"        : DARK["text"],
    "grid.color"        : DARK["grid"],
    "grid.linestyle"    : "--",
    "grid.alpha"        : 0.5,
    "font.family"       : "DejaVu Sans",
    "legend.facecolor"  : DARK["panel"],
    "legend.edgecolor"  : DARK["grid"],
    "axes.titleweight"  : "bold",
    "axes.titlesize"    : 13,
    "axes.titlecolor"   : DARK["text"],
    "figure.dpi"        : 110,
})

SEED = 42
np.random.seed(SEED)

print(" All imports successful.")
print(f"   NumPy   {np.__version__}")
print(f"   Pandas  {pd.__version__}")
import sklearn; print(f"   Sklearn {sklearn.__version__}")
import xgboost; print(f"   XGBoost {xgboost.__version__}")

 All imports successful.
   NumPy   2.4.3
   Pandas  3.0.1
   Sklearn 1.8.0
   XGBoost 3.2.0


## Data Loading 

In [5]:
# Load dataset 
DATA_PATH = "Bank_Transaction_Fraud_Detection.csv"

df = pd.read_csv("data/Bank_Transaction_Fraud_Detection.csv")

print("=" * 60)
print(f"  Dataset shape    : {df.shape[0]:,} rows  ×  {df.shape[1]} columns")
print(f"  Memory usage     : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Duplicate rows   : {df.duplicated().sum():,}")
print(f"  Missing values   : {df.isnull().sum().sum():,}")
print("=" * 60)

df.head(5)

  Dataset shape    : 200,000 rows  ×  24 columns
  Memory usage     : 261.7 MB
  Duplicate rows   : 0
  Missing values   : 0


,Customer_ID,Customer_Name,Gender,Age,State,City,Bank_Branch,Account_Type,Transaction_ID,Transaction_Date,...,Merchant_Category,Account_Balance,Transaction_Device,Transaction_Location,Device_Type,Is_Fraud,Transaction_Currency,Customer_Contact,Transaction_Description,Customer_Email
0,d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e,Osha Tella,Male,60,Kerala,Thiruvananthapuram,Thiruvananthapuram Branch,Savings,4fa3208f-9e23-42dc-b330-844829d0c12c,23-01-2025,...,Restaurant,74557.27,Voice Assistant,"Thiruvananthapuram, Kerala",POS,0,INR,+9198579XXXXXX,Bitcoin transaction,oshaXXXXX@XXXXX.com
1,7c14ad51-781a-4db9-b7bd-67439c175262,Hredhaan Khosla,Female,51,Maharashtra,Nashik,Nashik Branch,Business,c9de0c06-2c4c-40a9-97ed-3c7b8f97c79c,11-01-2025,...,Restaurant,74622.66,POS Mobile Device,"Nashik, Maharashtra",Desktop,0,INR,+9191074XXXXXX,Grocery delivery,hredhaanXXXX@XXXXXX.com
2,3a73a0e5-d4da-45aa-85f3-528413900a35,Ekani Nazareth,Male,20,Bihar,Bhagalpur,Bhagalpur Branch,Savings,e41c55f9-c016-4ff3-872b-cae72467c75c,25-01-2025,...,Groceries,66817.99,ATM,"Bhagalpur, Bihar",Desktop,0,INR,+9197745XXXXXX,Mutual fund investment,ekaniXXX@XXXXXX.com
3,7902f4ef-9050-4a79-857d-9c2ea3181940,Yamini Ramachandran,Female,57,Tamil Nadu,Chennai,Chennai Branch,Business,7f7ee11b-ff2c-45a3-802a-49bc47c02ecb,19-01-2025,...,Entertainment,58177.08,POS Mobile App,"Chennai, Tamil Nadu",Mobile,0,INR,+9195889XXXXXX,Food delivery,yaminiXXXXX@XXXXXXX.com
4,3a4bba70-d9a9-4c5f-8b92-1735fd8c19e9,Kritika Rege,Female,43,Punjab,Amritsar,Amritsar Branch,Savings,f8e6ac6f-81a1-4985-bf12-f60967d852ef,30-01-2025,...,Entertainment,16108.56,Virtual Card,"Amritsar, Punjab",Mobile,0,INR,+9195316XXXXXX,Debt repayment,kritikaXXXX@XXXXXX.com
